In [ ]:
!pip uninstall -y tf-nightly
!pip install tensorflow==2.16.1

import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)


In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Load dataset into pandas
train_df = pd.read_csv(train_file_path, sep="\t", names=["label", "message"])
test_df = pd.read_csv(test_file_path, sep="\t", names=["label", "message"])

# Convert labels to 0/1
train_df["label"] = train_df["label"].map({"ham": 0, "spam": 1})
test_df["label"] = test_df["label"].map({"ham": 0, "spam": 1})

# Split features and labels
train_messages = train_df["message"].values
train_labels = train_df["label"].values
test_messages = test_df["message"].values
test_labels = test_df["label"].values


In [ ]:
# Tokenize and vectorize text
max_features = 10000  # vocabulary size
sequence_length = 100

vectorize_layer = layers.TextVectorization(
    max_tokens=max_features,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Adapt vectorizer on training text
vectorize_layer.adapt(train_messages)

# Build model - deeper network
model = keras.Sequential([
    vectorize_layer,
    layers.Embedding(max_features, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64)),   # <-- stronger than pooling
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")
])

# Compile
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# Train (increase epochs to give it time to learn)
history = model.fit(
    train_messages,
    train_labels,
    epochs=15,
    batch_size=32,
    validation_data=(test_messages, test_labels),
    verbose=1
)


In [ ]:
# function to predict messages based on model
def predict_message(pred_text):
    # Ensure input goes through vectorizer
    input_text = tf.constant([pred_text])
    prob = model.predict(input_text, verbose=0)[0][0]

    label = "spam" if prob >= 0.5 else "ham"
    return [float(prob), label]

# quick test
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)


In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
